> **ARCHIVED research notebook** - kept for provenance and understanding only.
> It is **superseded by the `cloaq` package** (`src/cloaq/`) and may contain known bugs; see [`research/legacy-notebooks/README.md`](../README.md).
> Saved outputs were cleared and local paths removed. QASMBench paths are relative to your own checkout.

In [ ]:
from qiskit import QuantumCircuit
from qiskit import transpile
from qiskit.visualization import plot_histogram
from qiskit_ibm_runtime import SamplerV2
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
 
# Get a fake backend from the fake provider
backend = FakeManilaV2()

In [ ]:
#from qiskit_ibm_runtime import QiskitRuntimeService
#from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

#backend = QiskitRuntimeService().least_busy(operational=True, simulator=False)
#pass_Manager = generate_preset_pass_manager(optimization_level=1, backend=backend)


In [ ]:
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.circuit import QuantumCircuit, Clbit
from collections import OrderedDict

def remove_idle_qwires(circuit):
    dag = circuit_to_dag(circuit)
    idle_qubits = list(dag.idle_wires())

    # Identify active qubits
    active_qubits = [q for q in dag.qubits if q not in idle_qubits]

    # Create a new circuit with same number of classical bits
    new_circuit = QuantumCircuit(len(active_qubits), circuit.num_clbits)

    # Map qubits: original → new
    qubit_map = {q: new_circuit.qubits[i] for i, q in enumerate(active_qubits)}

    # Map clbits (1-to-1 since all classical bits are retained)
    clbit_map = {old: new for old, new in zip(circuit.clbits, new_circuit.clbits)}

    for instr, qargs, cargs in circuit.data:
        # Only include instructions on active qubits
        if all(q in qubit_map for q in qargs):
            new_qargs = [qubit_map[q] for q in qargs]
            new_cargs = [clbit_map[c] for c in cargs]
            new_circuit.append(instr, new_qargs, new_cargs)

    return new_circuit


In [ ]:
import numpy as np
import random
from qiskit import QuantumCircuit

def prepend_U3_gates(circuit):
    total_qubits = circuit.num_qubits
    angles = [i * (np.pi / 4) for i in range(9)] 
    U3_circuit = QuantumCircuit(total_qubits)

    for qubit in range(total_qubits):
        theta = random.choice(angles)
        phi = random.choice(angles)
        lam = random.choice(angles)
        U3_circuit.u(theta, phi, lam, qubit)

        full_circuit = U3_circuit.compose(circuit)
    return full_circuit

In [ ]:
import matplotlib.pyplot as plt
import qiskit
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import random
import numpy as np

key = "011"

#qc = QuantumCircuit.from_qasm_file('./QASMBench/small/adder_n4/adder_n4.qasm')
qc = QuantumCircuit.from_qasm_file('QASMBench/small/basis_trotter_n4/basis_trotter_n4.qasm')
qc = prepend_U3_gates(qc)

# qc = QuantumCircuit.from_qasm_file('QASMBench/small/fredkin_n3/fredkin_n3.qasm')
# qc = QuantumCircuit.from_qasm_file('QASMBench/small/basis_change_n3/basis_change_n3.qasm')
# qc = QuantumCircuit.from_qasm_file('QASMBench/small/wstate_n3/wstate_n3.qasm')
# Draw circuit
qc.draw('mpl')




In [ ]:
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

#backend = AerSimulator()

# Transpile the circuit using the pass manager
transpiled_circuit = transpile(qc,backend)

# Execute the transpiled circuit on the backend
job_sim = backend.run(transpiled_circuit, shots=1024)

# Grab the results from the job
result_sim = job_sim.result()

# Get the measurement counts
counts_unlocked = result_sim.get_counts(transpiled_circuit)
print(counts_unlocked)
plot_histogram(counts_unlocked)

In [ ]:
# Draw Transpiled Ciruit
transpiled_circuit_simp = remove_idle_qwires(transpiled_circuit)
transpiled_circuit_simp.draw('mpl')  

Locking

In [ ]:
"Modify this function instead of creating a new one to work with a standard"
"Also can leave current phase gate"

from qiskit import QuantumCircuit
from qiskit.circuit import Barrier

def transform_circuit(qc):
    """
    Transforms a quantum circuit such that each layer contains either:
    - Only phase gates (Rz, P, S, T, etc.), or
    - Only non-phase gates (X, Y, Z, H, CX, etc.).
    Adds barriers before and after phase gate layers.

    Args:
        qc (QuantumCircuit): The input quantum circuit.

    Returns:
        QuantumCircuit: The transformed quantum circuit.
    """
    # List of phase gates (you can extend this list as needed)
    phase_gates = {'rz', 'p', 's', 't', 'sx', 'sdg', 'tdg'}

    # Initialize a new circuit
    new_qc = QuantumCircuit(*qc.qregs, *qc.cregs)

    # Temporary lists to hold gates for the current layer
    current_layer = []
    is_phase_layer = None  # Tracks whether the current layer is a phase gate layer

    # Iterate through all gates in the circuit
    for instruction in qc:
        gate_name = instruction.operation.name.lower()  # Get the gate name in lowercase
        qubits = instruction.qubits
        clbits = instruction.clbits

        # Determine if the gate is a phase gate
        is_phase_gate = gate_name in phase_gates

        # If the current layer type changes, add it to the new circuit
        if is_phase_layer is not None and is_phase_gate != is_phase_layer:
            # Add the current layer to the new circuit
            for gate in current_layer:
                new_qc.append(gate)
            # Add barriers if the current layer was a phase gate layer
            if is_phase_layer:
                new_qc.barrier(*qc.qubits)
            # Reset the current layer
            current_layer = []
            is_phase_layer = None

        # Start a new layer if necessary
        if is_phase_layer is None:
            is_phase_layer = is_phase_gate
            if is_phase_layer:
                new_qc.barrier(*qc.qubits)  # Add barrier before phase gate layer

        # Add the gate to the current layer
        current_layer.append(instruction)

    # Add the last layer to the new circuit
    for gate in current_layer:
        new_qc.append(gate)
    if is_phase_layer:
        new_qc.barrier(*qc.qubits)  # Add barrier after the last phase gate layer

    return new_qc

In [ ]:
converted_circuit = transform_circuit(qc)
converted_circuit.draw('mpl')

In [ ]:
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

#backend = AerSimulator()

# Transpile the circuit using the pass manager
transpiled_circuit = transpile(converted_circuit, backend)

# Execute the transpiled circuit on the backend
job_sim = backend.run(transpiled_circuit, shots=1024)

# Grab the results from the job
result_sim = job_sim.result()

# Get the measurement counts
counts_original = result_sim.get_counts(transpiled_circuit)
print(counts_original)
plot_histogram(counts_original)

In [ ]:
# Draw Transpiled Ciruit
transpiled_circuit_simp = remove_idle_qwires(transpiled_circuit)
transpiled_circuit_simp.draw('mpl')  

In [ ]:
from math import pi

def key_phase(key):
    match key:
        case '000':
            return 0.25*pi
        case '001':
            return 0.5*pi
        case '010':
            return 0.75*pi
        case '011':
            return 1*pi
        case '100':
            return 1.25*pi
        case '101':
            return 1.5*pi
        case '110':
            return 1.75*pi
        case '111':
            return 2*pi

In [ ]:
from math import pi
from qiskit import QuantumCircuit

# def replace_with_phase_gates(circuit, dict_for_locking):
def replace_with_phase_gates(circuit):
    """
    Replace Z, S, T, and their inverse gates with custom phase gates in the QuantumCircuit.

    Parameters:
        circuit (QuantumCircuit): The original quantum circuit.

    Returns:
        QuantumCircuit: The modified quantum circuit with phase gates.
    """
    offset = key_phase(key)
    # Create a new circuit with the same number of qubits
    new_circuit = QuantumCircuit(circuit.num_qubits, circuit.num_clbits)
    
    # Iterate through the instructions in the circuit
    for instruction in circuit.data:
        gate, qubits, clbits = instruction
        if gate.name == "z":
            # Z gate is equivalent to a phase gate with pi
            new_circuit.p((pi)+offset, qubits[0])
        elif gate.name == "s":
            # S gate is equivalent to a phase gate with pi/2
            new_circuit.p((pi/2)+offset, qubits[0])
        elif gate.name == "t":
            # T gate is equivalent to a phase gate with pi/4
            new_circuit.p((pi/4)+offset, qubits[0])
        elif gate.name == "sdg":
            # S inverse (Sdg) is equivalent to a phase gate with -pi/2
            new_circuit.p((-pi/2)+offset, qubits[0])
        elif gate.name == "tdg":
            # T inverse (Tdg) is equivalent to a phase gate with -pi/4
            new_circuit.p((-pi/4)+offset, qubits[0])
        else:
            if gate.name == "x":
                new_circuit.x(qubits[0])
            elif gate.name == "h":
                new_circuit.h(qubits[0])
            elif gate.name == "cx":
                new_circuit.cx(qubits[0], qubits[1])
            elif gate.name == "measure":
                new_circuit.measure(qubits[0], clbits[0])
    return transform_circuit(new_circuit)

In [ ]:
phase_converted_circuit = replace_with_phase_gates(converted_circuit)
# phase_converted_circuit.draw('mpl')
obfuscated_circuit = prepend_U3_gates(phase_converted_circuit)
obfuscated_circuit.draw('mpl')

In [ ]:
from qiskit import QuantumCircuit

def add_measurements_from_info(circuit, measurement_info):
    """
    Adds measurement gates to a quantum circuit based on the provided measurement information.

    Parameters:
    circuit (QuantumCircuit): The quantum circuit to which measurements will be added.
    measurement_info (dict): A dictionary containing measurement information, as returned by `get_measurement_info`.

    Returns:
    QuantumCircuit: The modified quantum circuit with measurement gates added.
    """
    # Iterate over the measurement information
    for qubit, info in measurement_info.items():
        # Extract the qubit and classical bit indices
        qubit_index = info['measured_qubit']
        clbit_index = info['classical_bit']
        
        # Add the measurement gate to the circuit
        circuit.measure(qubit_index, clbit_index)
    
    return circuit

def remove_all_measurements(circuit):
    """
    Removes all measurement gates from a quantum circuit.

    Parameters:
    circuit (QuantumCircuit): The quantum circuit from which measurements will be removed.

    Returns:
    QuantumCircuit: A new quantum circuit with all measurement gates removed.
    """
    # Create a new circuit with the same number of qubits and classical bits
    new_circuit = QuantumCircuit(circuit.num_qubits, circuit.num_clbits)
    
    # Iterate through all operations in the original circuit
    for instruction, qubits, clbits in circuit.data:
        # Skip measurement gates
        if instruction.name != 'measure':
            # Add the non-measurement operation to the new circuit
            new_circuit.append(instruction, qubits, clbits)
    
    return new_circuit

def get_measurement_info(circuit):
    """
    Analyzes a quantum circuit and provides information about measurement gates.

    Parameters:
    circuit (QuantumCircuit): The quantum circuit to analyze.

    Returns:
    dict: A dictionary containing information about the measurement gates.
          The keys are qubit indices, and the values are dictionaries with details
          about the measurement gate and the classical bit where the result is stored.
    """
    measurement_info = {}
    
    # Iterate through all operations in the circuit
    for instruction, qubits, clbits in circuit.data:
        if instruction.name == 'measure':  # Check if the operation is a measurement gate
            # Get the index of the qubit being measured
            qubit_index = circuit.find_bit(qubits[0]).index
            # Get the index of the classical bit where the result is stored
            clbit_index = circuit.find_bit(clbits[0]).index
            
            # Store the measurement information
            measurement_info[qubit_index] = {
                'measured_qubit': qubit_index,
                'measurement_gate': 'M',  # 'M' represents the measurement gate
                'classical_bit': clbit_index  # The classical bit where the result is stored
            }
    
    return measurement_info

measurement_info = get_measurement_info(phase_converted_circuit)
# Print the measurement information
print("Measurement Information:")
for qubit, info in measurement_info.items():
    print(f"Qubit {info['measured_qubit']} is measured using gate '{info['measurement_gate']}' "
          f"and the result is stored in classical bit {info['classical_bit']}.")

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import SGate, TGate, PhaseGate, ZGate
import random

def add_random_phase_layers_with_reassembly(circuit, num_layers, num_splits):
    """
    Dissects a quantum circuit into parts, adds layers of random phase gates
    sandwiched between barriers, and reassembles the circuit.

    Parameters:
    circuit (QuantumCircuit): The quantum circuit to modify.
    num_layers (int): The number of random phase gate layers to add at each split.
    num_splits (int): The number of times to split the circuit and add phase gates.

    Returns:
    QuantumCircuit: The modified quantum circuit with random phase gates and barriers.
    """
    phases = [0.25 * pi, 0.5 * pi, 0.75 * pi, 1 * pi, 1.25 * pi, 1.5 * pi, 1.75 * pi, 2 * pi]

    # List of standard phase gates
    phase_gates = [PhaseGate(random.choice(phases))] #[PhaseGate(random.uniform(0, 2 * 3.14159))]#[SGate(), TGate(), ZGate()]
    
    # Get the number of qubits in the circuit
    num_qubits = circuit.num_qubits

    circuit = prepend_U3_gates(circuit)
    
    # Create a new circuit to build the modified version
    new_circuit = QuantumCircuit(num_qubits)
    
    # Convert the circuit's data (operations) into a list for easy manipulation
    operations = list(circuit.data)
    
    # Calculate the split points
    split_points = [len(operations) // (num_splits + 1) * (i + 1) for i in range(num_splits)]
    
    # Iterate through the original circuit's operations
    for i, (instruction, qubits, clbits) in enumerate(operations):
        # Add the original operation to the new circuit
        new_circuit.append(instruction, qubits, clbits)
        
        # Check if we are at a split point
        if i + 1 in split_points:
            # Add layers of random phase gates sandwiched between barriers
            for _ in range(num_layers):
                # Add a barrier before the phase gates
                new_circuit.barrier(range(num_qubits))
                
                # Add random phase gates to each qubit
                for qubit in range(num_qubits):
                    gate = random.choice(phase_gates)
                    new_circuit.append(gate, [qubit])
                
                # Add a barrier after the phase gates
                new_circuit.barrier(range(num_qubits))

    new_circuit.measure_all()
    return new_circuit

In [ ]:
random_layer_added_qc = add_random_phase_layers_with_reassembly(remove_all_measurements(phase_converted_circuit), 1,2)
random_layer_added_qc.draw('mpl')


In [ ]:
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

# Use AerSimulator as the backend
backend = AerSimulator()

# Transpile the circuit using the pass manager
transpiled_circuit = transpile(random_layer_added_qc, backend)

# Execute the transpiled circuit on the backend
job_sim = backend.run(transpiled_circuit, shots=1024)

# Grab the results from the job
result_sim = job_sim.result()

# Get the measurement counts
counts_locked = result_sim.get_counts(transpiled_circuit)
print(counts_locked)
plot_histogram(counts_locked)

In [ ]:
transpiled_circuit_simp = remove_idle_qwires(transpiled_circuit)
transpiled_circuit_simp.draw('mpl')  

TVD Locked

In [ ]:
import numpy as np
def total_variation_distance_from_counts(counts1, counts2, shots = 1024):
    """
    Calculate the Total Variation Distance (TVD) between two probability distributions
    derived from Qiskit counts.

    Parameters:
    counts1 (dict): The counts from the first circuit.
    counts2 (dict): The counts from the second circuit.

    Returns:
    float: The Total Variation Distance between the two distributions.
    """
    # Get all possible keys (states) from both counts
    all_states = set(counts1.keys()).union(set(counts2.keys()))
    
    # Normalize counts to get probability distributions
    total_counts1 = sum(counts1.values())
    total_counts2 = sum(counts2.values())
    
    # Initialize probability distributions
    p = np.zeros(len(all_states))
    q = np.zeros(len(all_states))
   
    # Map states to indices
    state_to_index = {state: i for i, state in enumerate(all_states)}
    
    # Fill probability distributions
    for state, count in counts1.items():
        p[state_to_index[state]] = count 
    for state, count in counts2.items():
        q[state_to_index[state]] = count 
  
    # Calculate the Total Variation Distance
    tvd = (1/(2*1024)) * np.sum(np.abs(p - q))
    return tvd

tvd = total_variation_distance_from_counts(counts_original, counts_locked)
print(f"Total Variation Distance: {tvd:.6f}")
print(f"key: {key}")

Unlocking

In [ ]:
# from qiskit import QuantumCircuit
# from qiskit.circuit.library import RZGate, PhaseGate
# from qiskit.circuit import Barrier

# def subtract_phase_offset(circuit: QuantumCircuit, offset: float) -> QuantumCircuit:
#     """
#     Subtracts a phase offset from all RZGate and PhaseGate instructions within barriers.

#     Args:
#         circuit (QuantumCircuit): The input quantum circuit.
#         offset (float): The phase offset to subtract.

#     Returns:
#         QuantumCircuit: The modified quantum circuit.
#     """
#     # Create a new circuit with the same number of qubits and classical bits
#     new_circuit = QuantumCircuit(circuit.num_qubits, circuit.num_clbits)

#     # Copy over all the registers from the original circuit
#     for qreg in circuit.qregs:
#         new_circuit.add_register(qreg)
#     for creg in circuit.cregs:
#         new_circuit.add_register(creg)

#     # Flag to indicate if we are inside a barrier
#     inside_barrier = False

#     # Iterate over the instructions in the circuit
#     for instruction, qargs, cargs in circuit.data:
#         # Check if the instruction is a barrier
#         if isinstance(instruction, Barrier):
#             inside_barrier = not inside_barrier  # Toggle the flag
#             new_circuit.append(instruction, qargs, cargs)  # Add the barrier to the new circuit
#         elif inside_barrier and (isinstance(instruction, RZGate) or isinstance(instruction, PhaseGate)):
#             # If inside a barrier and the instruction is a phase gate, modify its phase
#             original_phase = instruction.params[0]
#             new_phase = original_phase - offset
#             # Add the modified phase gate to the new circuit
#             if isinstance(instruction, RZGate):
#                 new_circuit.rz(new_phase, *qargs)
#             elif isinstance(instruction, PhaseGate):
#                 new_circuit.p(new_phase, *qargs)
#         else:
#             # Otherwise, add the instruction as is
#             new_circuit.append(instruction, qargs, cargs)

#     return new_circuit  # Return the modified circuit

from qiskit import QuantumCircuit
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import RemoveBarriers, DAGFixedPoint
from qiskit.converters import circuit_to_dag, dag_to_circuit

def remove_gates_inside_barriers(circuit):
    # Convert the circuit to a DAG (Directed Acyclic Graph)
    dag = circuit_to_dag(circuit)
    
    # Iterate over all nodes in the DAG
    for node in dag.op_nodes():
        # Check if the node is a barrier
        if node.name == 'barrier':
            # Get the qubits involved in the barrier
            barrier_qubits = node.qargs
            
            # Remove all gates that are inside the barrier
            for successor in list(dag.successors(node)):
                if all(qubit in barrier_qubits for qubit in successor.qargs):
                    print(f"Removing gate {successor.name} inside barrier")
                    # dag.remove_op_node(successor)
    
    # Convert the modified DAG back to a QuantumCircuit
    new_circuit = dag_to_circuit(dag)
    
    return new_circuit

def modify_rz_inside_barriers(circuit, phase_shift):
    """
    Modify the phase of Rz gates inside barriers by adding a phase shift.
    
    Args:
        circuit (QuantumCircuit): The input quantum circuit.
        phase_shift (float): The phase shift to apply to Rz gates inside barriers.
    
    Returns:
        QuantumCircuit: The modified quantum circuit.
    """
    # Convert the circuit to a DAG (Directed Acyclic Graph)
    dag = circuit_to_dag(circuit)
    
    # Track whether we are inside a barrier
    inside_barrier = False
    
    # Iterate over all nodes in the DAG
    for node in dag.topological_op_nodes():
        # Check if the node is a barrier
        if node.name == 'barrier':
            inside_barrier = not inside_barrier  # Toggle the inside_barrier flag
            continue
        
        # If inside a barrier and the node is an Rz gate, modify its phase
        if inside_barrier and node.name == 'rz':
            # Get the current phase of the Rz gate
            current_phase = node.op.params[0]
            # Apply the phase shift
            new_phase = current_phase + phase_shift
            # Update the Rz gate with the new phase
            node.op.params[0] = 0
    
    # Convert the modified DAG back to a QuantumCircuit
    modified_circuit = dag_to_circuit(dag)
    
    return modified_circuit

In [ ]:
p_offset = key_phase("010")
modified_circuit = remove_gates_inside_barriers(transpiled_circuit_simp)
# modified_circuit = modify_rz_inside_barriers(transpiled_circuit_simp, -p_offset)

modified_circuit.draw('mpl')

In [ ]:
from qiskit import transpile 
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

# Use AerSimulator as the backend
backend = AerSimulator()

# Transpile the circuit using the pass manager
transpiled_circuit = transpile(modified_circuit,backend)

# Execute the transpiled circuit on the backend
job_sim = backend.run(transpiled_circuit, shots=1024)

# Grab the results from the job
result_sim = job_sim.result()

# Get the measurement counts
counts = result_sim.get_counts(transpiled_circuit)
print(counts)
plot_histogram(counts)


In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import PhaseGate
from qiskit.circuit import Barrier

def subtract_phase_offset(circuit: QuantumCircuit, offset: float) -> QuantumCircuit:
    """
    This function takes a quantum circuit, looks for barriers, and inside each barrier,
    it looks for phase gates and subtracts an offset from their phase.

    Args:
        circuit (QuantumCircuit): The input quantum circuit.
        offset (float): The offset to subtract from the phase of phase gates.

    Returns:
        QuantumCircuit: The modified quantum circuit.
    """
    # Create a new circuit to store the modified instructions
    new_circuit = QuantumCircuit(*circuit.qregs, *circuit.cregs)

    # Flag to indicate if we are inside a barrier
    inside_barrier = False

    # Iterate over the instructions in the circuit
    for instruction, qargs, cargs in circuit.data:
        # Check if the instruction is a barrier
        if isinstance(instruction, Barrier):
            inside_barrier = not inside_barrier  # Toggle the flag
            new_circuit.append(instruction, qargs, cargs)  # Add the barrier to the new circuit
        elif inside_barrier and isinstance(instruction, PhaseGate):
            # If inside a barrier and the instruction is a phase gate, modify its phase
            original_phase = instruction.params[0]
            new_phase = original_phase - offset
            new_circuit.p(new_phase, *qargs)  # Add the modified phase gate
        else:
            # Otherwise, add the instruction as is
            new_circuit.append(instruction, qargs, cargs)

    return new_circuit

# Example usage:
#qc = QuantumCircuit(2)
#qc.p(0.5, 0)  # Add a phase gate with phase 0.5
#qc.barrier()  # Add a barrier
#qc.p(0.8, 1)  # Add another phase gate with phase 0.8
#qc.barrier()  # Add another barrier

#print("Original Circuit:")
#print(qc)

# Subtract an offset of 0.2 from the phase gates within barriers
#modified_qc = subtract_phase_offset(qc, 0.2)

#print("\nModified Circuit:")
#print(modified_qc)

In [ ]:
# Load original circuit from QASM file 
qc_original = QuantumCircuit.from_qasm_file('./QASMBench/small/adder_n4/adder_n4.qasm')

# Create locked circuit by transforming
qc_locked = transform_circuit(qc_original)


# Setup simulator
shots = 1024
num_runs = 100
tvd_values = []

# Run circuits multiple times, compute TVD
for _ in range(num_runs):
    job_orig = backend.run(qc_original, shots=shots)
    result_orig = job_orig.result()
    counts_orig = result_orig.get_counts()

    job_lock = backend.run(qc_locked, shots=shots)
    result_lock = job_lock.result()
    counts_lock = result_lock.get_counts()

    tvd = total_variation_distance_from_counts(counts_orig, counts_lock, shots)
    tvd_values.append(tvd)

# Plot histogram of TVD values
plt.hist(tvd_values, bins=20, color='skyblue', edgecolor='black')
plt.title('Histogram of Total Variation Distances over 100 runs')
plt.xlabel('Total Variation Distance')
plt.ylabel('Frequency')
plt.show()